In [16]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path


In [17]:
df = pd.read_csv("../data/processed/station_hour_clean.csv")
df.head()

,Datetime,StationId,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,2017-11-25 09:00:00,AP001,104.00,148.5,1.93,23.00,13.75,9.80,0.1,15.3,117.62,0.30,10.40,0.23,155.0,Moderate
1,2017-11-25 10:00:00,AP001,94.50,142.0,1.33,16.25,9.75,9.65,0.1,17.0,136.23,0.28,7.10,0.15,159.0,Moderate
2,2017-11-25 11:00:00,AP001,82.75,126.5,1.47,14.83,9.07,9.70,0.1,15.4,149.92,0.20,4.55,0.08,173.0,Moderate
3,2017-11-25 12:00:00,AP001,79.00,124.0,5.30,21.15,15.53,9.40,0.1,15.4,156.80,0.20,4.00,0.00,184.0,Moderate
4,2017-11-25 13:00:00,AP001,79.00,124.0,5.30,21.15,15.53,9.40,0.1,15.4,156.80,0.20,4.00,0.00,184.0,Moderate


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 304644 entries, 0 to 304643
Data columns (total 16 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   Datetime    304644 non-null  object 
 1   StationId   304644 non-null  object 
 2   PM2.5       304644 non-null  float64
 3   PM10        304644 non-null  float64
 4   NO          304644 non-null  float64
 5   NO2         304644 non-null  float64
 6   NOx         304644 non-null  float64
 7   NH3         304644 non-null  float64
 8   CO          304644 non-null  float64
 9   SO2         304644 non-null  float64
 10  O3          304644 non-null  float64
 11  Benzene     304644 non-null  float64
 12  Toluene     304644 non-null  float64
 13  Xylene      304644 non-null  float64
 14  AQI         304644 non-null  float64
 15  AQI_Bucket  304644 non-null  object 
dtypes: float64(13), object(3)
memory usage: 37.2+ MB


In [26]:
feature_cols = [
    "PM2.5", "PM10", "NO", "NO2", "NOx",
    "NH3", "CO", "SO2", "O3",
    "Benzene", "Toluene", "Xylene"
]
target_col = "AQI"

In [20]:
df["Datetime"] = pd.to_datetime(df["Datetime"])

In [21]:
df = (
    df.sort_values(["StationId", "Datetime"])
      .set_index("Datetime")
)

In [22]:
# --- Fitur waktu (pola siklus harian/mingguan) ---
df["hour"]       = df.index.hour
df["dayofweek"]  = df.index.dayofweek   # 0=Senin, 6=Minggu
df["month"]      = df.index.month
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

# Encode jam dan bulan sebagai sin/cos agar model tahu sifat siklisnya
df["hour_sin"]   = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"]   = np.cos(2 * np.pi * df["hour"] / 24)
df["month_sin"]  = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"]  = np.cos(2 * np.pi * df["month"] / 12)

# --- Rolling features per stasiun (rata-rata 3 jam, 6 jam, 24 jam) ---
key_pollutants = ["PM2.5", "PM10", "NO2", "CO", "O3"]

for col in key_pollutants:
    for window in [3, 6, 24]:
        df[f"{col}_roll{window}"] = (
            df.groupby("StationId")[col]
            .transform(lambda x: x.rolling(window, min_periods=1).mean())
        )

# --- Lag features (1 jam dan 3 jam sebelumnya untuk PM2.5 dan AQI) ---
for lag in [1, 3]:
    df[f"PM2.5_lag{lag}"] = (
        df.groupby("StationId")["PM2.5"]
        .transform(lambda x: x.shift(lag))
    )
    df[f"AQI_lag{lag}"] = (
        df.groupby("StationId")[target_col]
        .transform(lambda x: x.shift(lag))
    )

# Hapus baris yang lag-nya NaN (awal tiap stasiun)
lag_cols = [c for c in df.columns if "lag" in c]
df = df.dropna(subset=lag_cols)

print(f"Shape setelah feature engineering: {df.shape}")
print("\nKolom baru:")
new_cols = [c for c in df.columns if any(x in c for x in ["roll", "lag", "sin", "cos", "hour", "month", "weekend"])]
print(new_cols)

Shape setelah feature engineering: (304584, 42)

Kolom baru:
['hour', 'month', 'is_weekend', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'PM2.5_roll3', 'PM2.5_roll6', 'PM2.5_roll24', 'PM10_roll3', 'PM10_roll6', 'PM10_roll24', 'NO2_roll3', 'NO2_roll6', 'NO2_roll24', 'CO_roll3', 'CO_roll6', 'CO_roll24', 'O3_roll3', 'O3_roll6', 'O3_roll24', 'PM2.5_lag1', 'AQI_lag1', 'PM2.5_lag3', 'AQI_lag3']


In [27]:
temporal_cols = [
    "hour_sin", "hour_cos", "month_sin", "month_cos", "is_weekend"
]

rolling_cols  = [c for c in df.columns if "roll" in c]
lag_cols_feat = [c for c in df.columns if "lag" in c]

# Gabungkan semua fitur untuk LSTM
feature_cols_lstm = feature_cols + temporal_cols + rolling_cols + lag_cols_feat

print(f"Total fitur untuk LSTM: {len(feature_cols_lstm)}")
print(feature_cols_lstm)

Total fitur untuk LSTM: 36
['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'is_weekend', 'PM2.5_roll3', 'PM2.5_roll6', 'PM2.5_roll24', 'PM10_roll3', 'PM10_roll6', 'PM10_roll24', 'NO2_roll3', 'NO2_roll6', 'NO2_roll24', 'CO_roll3', 'CO_roll6', 'CO_roll24', 'O3_roll3', 'O3_roll6', 'O3_roll24', 'PM2.5_lag1', 'AQI_lag1', 'PM2.5_lag3', 'AQI_lag3']


In [32]:
df.head()

,StationId,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,...,CO_roll3,CO_roll6,CO_roll24,O3_roll3,O3_roll6,O3_roll24,PM2.5_lag1,AQI_lag1,PM2.5_lag3,AQI_lag3
Datetime,,,,,,,,,,,,,,,,,,,,,
2017-11-25 12:00:00,AP001,79.00,124.00,5.30,21.15,15.53,9.40,0.1,15.40,156.80,...,0.1,0.1,0.1,147.650000,140.142500,140.142500,82.75,173.0,104.00,155.0
2017-11-25 13:00:00,AP001,79.00,124.00,5.30,21.15,15.53,9.40,0.1,15.40,156.80,...,0.1,0.1,0.1,154.506667,143.474000,143.474000,79.00,184.0,94.50,159.0
2017-11-25 14:00:00,AP001,68.50,117.00,1.35,13.60,8.35,7.40,0.1,21.80,161.70,...,0.1,0.1,0.1,158.433333,146.511667,146.511667,79.00,184.0,82.75,173.0
2017-11-25 15:00:00,AP001,69.25,112.25,1.52,11.80,7.55,9.25,0.1,21.38,161.68,...,0.1,0.1,0.1,160.060000,153.855000,148.678571,68.50,191.0,79.00,184.0
2017-11-25 16:00:00,AP001,70.00,107.00,2.80,30.33,18.40,6.15,0.1,18.90,147.97,...,0.1,0.1,0.1,157.116667,155.811667,148.590000,69.25,191.0,79.00,184.0


In [33]:
df_final = df.reset_index()

In [35]:
df_final.head()

,Datetime,StationId,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,...,CO_roll3,CO_roll6,CO_roll24,O3_roll3,O3_roll6,O3_roll24,PM2.5_lag1,AQI_lag1,PM2.5_lag3,AQI_lag3
0,2017-11-25 12:00:00,AP001,79.00,124.00,5.30,21.15,15.53,9.40,0.1,15.40,...,0.1,0.1,0.1,147.650000,140.142500,140.142500,82.75,173.0,104.00,155.0
1,2017-11-25 13:00:00,AP001,79.00,124.00,5.30,21.15,15.53,9.40,0.1,15.40,...,0.1,0.1,0.1,154.506667,143.474000,143.474000,79.00,184.0,94.50,159.0
2,2017-11-25 14:00:00,AP001,68.50,117.00,1.35,13.60,8.35,7.40,0.1,21.80,...,0.1,0.1,0.1,158.433333,146.511667,146.511667,79.00,184.0,82.75,173.0
3,2017-11-25 15:00:00,AP001,69.25,112.25,1.52,11.80,7.55,9.25,0.1,21.38,...,0.1,0.1,0.1,160.060000,153.855000,148.678571,68.50,191.0,79.00,184.0
4,2017-11-25 16:00:00,AP001,70.00,107.00,2.80,30.33,18.40,6.15,0.1,18.90,...,0.1,0.1,0.1,157.116667,155.811667,148.590000,69.25,191.0,79.00,184.0


In [37]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "final_dataset.csv"

df_final.to_csv(output_path, index=False)

print(f"Dataset berhasil disimpan di: {output_path}")

Dataset berhasil disimpan di: ..\data\processed\final_dataset.csv
